## Name: Asit Jain
## Roll No: M25DE1049
## Assignment 3 - MLBD

# Part 5: Learning-Based Recommender Systems
## Task 8: Content-Based Filtering with a Neural Network

Neural network for content-based filtering using movie metadata and user genre preferences.

```
User Features  -> Dense -> User Embedding  \
                                            -> Combine -> Predict Rating
Movie Features -> Dense -> Movie Embedding /
```

**Dataset:** MovieLens ml-latest-small

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'pandas', 'numpy', 'scikit-learn', 'torch', '-q'])

import os, zipfile, urllib.request, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
print(f'PyTorch {torch.__version__} - All imports successful!')

PyTorch 2.10.0+cpu - All imports successful!


### Step 1: Load and Process the Dataset

In [2]:
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
zip_path = 'ml-latest-small.zip'
extract_dir = 'ml-latest-small'
if not os.path.exists(extract_dir):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z: z.extractall('.')

movies = pd.read_csv(os.path.join(extract_dir, 'movies.csv'))
ratings = pd.read_csv(os.path.join(extract_dir, 'ratings.csv'))
print(f'Movies: {len(movies)}, Ratings: {len(ratings)}')

Movies: 9742, Ratings: 100836


### Step 2: Extract Movie Metadata Features

For each movie: one-hot genres, release year, average rating.

In [3]:
# Extract release year from title
def extract_year(title):
    match = re.search(r'\((\d{4})\)', str(title))
    return int(match.group(1)) if match else np.nan

movies['year'] = movies['title'].apply(extract_year)
movies['year'] = movies['year'].fillna(movies['year'].median())

# One-hot encode genres
all_genres = sorted(set(g for gs in movies['genres'] for g in gs.split('|') if g != '(no genres listed)'))
print(f'Genres ({len(all_genres)}): {all_genres}')

for genre in all_genres:
    movies[f'g_{genre}'] = movies['genres'].apply(lambda x: 1.0 if genre in x.split('|') else 0.0)

# Average rating per movie
movie_avg = ratings.groupby('movieId')['rating'].mean()
movies['avg_rating'] = movies['movieId'].map(movie_avg).fillna(ratings['rating'].mean())

# Build movie feature matrix
genre_cols = [f'g_{g}' for g in all_genres]
scaler_m = StandardScaler()
movies[['year_n', 'avg_n']] = scaler_m.fit_transform(movies[['year', 'avg_rating']])
movie_feat_cols = genre_cols + ['year_n', 'avg_n']
movie_features = movies.set_index('movieId')[movie_feat_cols]
print(f'Movie feature matrix: {movie_features.shape}')

Genres (19): ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
Movie feature matrix: (9742, 21)


### Step 3: Extract User Features (Average Rating per Genre)

In [4]:
ratings_g = ratings.merge(movies[['movieId'] + genre_cols], on='movieId')

user_genre_prefs = {}
for uid, group in ratings_g.groupby('userId'):
    prefs = []
    for g in genre_cols:
        mask = group[g] == 1.0
        prefs.append(group.loc[mask, 'rating'].mean() if mask.sum() > 0 else 0.0)
    user_genre_prefs[uid] = prefs

user_features_df = pd.DataFrame.from_dict(user_genre_prefs, orient='index', columns=genre_cols)
scaler_u = StandardScaler()
user_features = pd.DataFrame(scaler_u.fit_transform(user_features_df),
    index=user_features_df.index, columns=user_features_df.columns)
print(f'User feature matrix: {user_features.shape}')

User feature matrix: (610, 19)


### Step 4: Prepare PyTorch Datasets

In [5]:
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=42)
valid_u = set(user_features.index)
valid_m = set(movie_features.index)
train_df = train_df[train_df['userId'].isin(valid_u) & train_df['movieId'].isin(valid_m)].copy()
test_df = test_df[test_df['userId'].isin(valid_u) & test_df['movieId'].isin(valid_m)].copy()
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

n_uf = user_features.shape[1]
n_mf = movie_features.shape[1]

class RatingDS(Dataset):
    def __init__(self, df, uf, mf):
        self.u = torch.FloatTensor(np.array([uf.loc[r['userId']].values for _, r in df.iterrows()]))
        self.m = torch.FloatTensor(np.array([mf.loc[r['movieId']].values for _, r in df.iterrows()]))
        self.r = torch.FloatTensor(df['rating'].values)
    def __len__(self): return len(self.r)
    def __getitem__(self, i): return self.u[i], self.m[i], self.r[i]

train_ds = RatingDS(train_df, user_features, movie_features)
test_ds = RatingDS(test_df, user_features, movie_features)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=512)
print('Datasets ready.')

Train: 80668, Test: 20168


C:\Users\Avinash Singh\AppData\Local\Temp\ipykernel_23544\3751108600.py:15: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  self.r = torch.FloatTensor(df['rating'].values)


Datasets ready.


### Step 5: Define Two-Branch Neural Network

In [6]:
class ContentNN(nn.Module):
    def __init__(self, n_uf, n_mf, emb=32):
        super().__init__()
        self.user_br = nn.Sequential(
            nn.Linear(n_uf, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, emb), nn.ReLU())
        self.movie_br = nn.Sequential(
            nn.Linear(n_mf, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, emb), nn.ReLU())
        self.pred = nn.Sequential(
            nn.Linear(emb*2, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, u, m):
        return self.pred(torch.cat([self.user_br(u), self.movie_br(m)], dim=1)).squeeze(1)

model = ContentNN(n_uf, n_mf)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

ContentNN(
  (user_br): Sequential(
    (0): Linear(in_features=19, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
  )
  (movie_br): Sequential(
    (0): Linear(in_features=21, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
  )
  (pred): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Linear(in_features=32, out_features=1, bias=True)
  )
)
Parameters: 13,121


### Step 6: Train the Model (MSE Loss, Adam, Early Stopping)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.001)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
criterion = nn.MSELoss()

best_val, best_state, patience = float('inf'), None, 0
for epoch in range(30):
    model.train()
    tloss = 0
    for u, m, r in train_loader:
        u, m, r = u.to(device), m.to(device), r.to(device)
        opt.zero_grad()
        loss = criterion(model(u, m), r)
        loss.backward(); opt.step()
        tloss += loss.item() * len(r)
    tloss /= len(train_ds)

    model.eval()
    vloss = 0
    with torch.no_grad():
        for u, m, r in test_loader:
            u, m, r = u.to(device), m.to(device), r.to(device)
            vloss += criterion(model(u, m), r).item() * len(r)
    vloss /= len(test_ds)
    sched.step(vloss)

    if vloss < best_val:
        best_val = vloss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience = 0
    else:
        patience += 1

    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:>2d} | Train MSE: {tloss:.4f} | Val MSE: {vloss:.4f} | Val RMSE: {np.sqrt(vloss):.4f}')
    if patience >= 7:
        print(f'Early stopping at epoch {epoch+1}')
        break

model.load_state_dict(best_state)
model = model.to(device)
print(f'Best Val RMSE: {np.sqrt(best_val):.4f}')

Epoch  1 | Train MSE: 1.8722 | Val MSE: 0.6877 | Val RMSE: 0.8293
Epoch  5 | Train MSE: 0.6976 | Val MSE: 0.7029 | Val RMSE: 0.8384
Epoch 10 | Train MSE: 0.6501 | Val MSE: 0.7226 | Val RMSE: 0.8501
Early stopping at epoch 13
Best Val RMSE: 0.8143


### Step 7: Evaluate - RMSE, Precision@K, Recall@K

In [8]:
model.eval()
preds_all, actuals_all = [], []
with torch.no_grad():
    for u, m, r in test_loader:
        p = model(u.to(device), m.to(device)).cpu().numpy()
        preds_all.extend(p); actuals_all.extend(r.numpy())

preds_all = np.clip(preds_all, 0.5, 5.0)
nn_rmse = np.sqrt(mean_squared_error(actuals_all, preds_all))
print(f'Neural CBF RMSE: {nn_rmse:.4f}')

test_eval = test_df.copy()
test_eval['nn_pred'] = preds_all

def prec_rec(df, col, k=10, thr=4.0):
    ps, rs = [], []
    for uid, g in df.groupby('userId'):
        rel = set(g[g['rating'] >= thr]['movieId'])
        if not rel: continue
        top = g.nlargest(k, col)
        hits = set(top['movieId']) & rel
        ps.append(len(hits)/k); rs.append(len(hits)/len(rel))
    return np.mean(ps) if ps else 0, np.mean(rs) if rs else 0

nn_p, nn_r = prec_rec(test_eval, 'nn_pred')
print(f'Precision@10: {nn_p:.4f}')
print(f'Recall@10:    {nn_r:.4f}')

Neural CBF RMSE: 0.8142
Precision@10: 0.5900
Recall@10:    0.6933


### Step 8: Compare with Traditional TF-IDF CBF

In [9]:
# TF-IDF CBF baseline (from Task 2)
movies_cbf = movies.copy()
movies_cbf['genre_text'] = movies_cbf['genres'].str.replace('|', ' ', regex=False).replace('(no genres listed)', '')
tfidf = TfidfVectorizer(stop_words='english')
tfidf_mat = tfidf.fit_transform(movies_cbf['genre_text'])
mid_idx = pd.Series(movies_cbf.index, index=movies_cbf['movieId'])

cbf_profiles = {}
for uid, group in train_df.groupby('userId'):
    valid = group[group['movieId'].isin(mid_idx.index)]
    if valid.empty: continue
    vecs = tfidf_mat[mid_idx[valid['movieId']].values].toarray()
    w = valid['rating'].values.reshape(-1, 1)
    cbf_profiles[uid] = (vecs * w).sum(axis=0) / w.sum()

g_avg = train_df['rating'].mean()
cbf_preds = []
for _, row in test_df.iterrows():
    uid, mid = row['userId'], row['movieId']
    if uid in cbf_profiles and mid in mid_idx.index:
        sim = cosine_similarity(cbf_profiles[uid].reshape(1,-1), tfidf_mat[mid_idx[mid]])[0,0]
        cbf_preds.append(0.5 + sim * 4.5)
    else:
        cbf_preds.append(g_avg)

cbf_preds = np.array(cbf_preds)
cbf_rmse = np.sqrt(mean_squared_error(test_df['rating'], cbf_preds))
test_eval['cbf_pred'] = cbf_preds
cbf_p, cbf_r = prec_rec(test_eval, 'cbf_pred')

print('=' * 55)
print('COMPARISON: Neural CBF vs Traditional TF-IDF CBF')
print('=' * 55)
print(f'{"Metric":<15s} | {"TF-IDF CBF":>12s} | {"Neural CBF":>12s}')
print('-' * 45)
print(f'{"RMSE":<15s} | {cbf_rmse:>12.4f} | {nn_rmse:>12.4f}')
print(f'{"Precision@10":<15s} | {cbf_p:>12.4f} | {nn_p:>12.4f}')
print(f'{"Recall@10":<15s} | {cbf_r:>12.4f} | {nn_r:>12.4f}')

COMPARISON: Neural CBF vs Traditional TF-IDF CBF
Metric          |   TF-IDF CBF |   Neural CBF
---------------------------------------------
RMSE            |       1.5120 |       0.8142
Precision@10    |       0.4908 |       0.5900
Recall@10       |       0.6388 |       0.6933


### Step 9: Top-N Recommendations

In [10]:
def recommend_nn(uid, model, uf, mf, movies_df, train_data, top_n=10):
    rated = set(train_data[train_data['userId'] == uid]['movieId'])
    cands = [m for m in movies_df['movieId'] if m not in rated and m in mf.index]
    if uid not in uf.index or not cands: return pd.DataFrame()
    u_t = torch.FloatTensor(uf.loc[uid].values).unsqueeze(0).repeat(len(cands), 1)
    m_t = torch.FloatTensor(np.array([mf.loc[m].values for m in cands]))
    model.eval()
    with torch.no_grad():
        p = model(u_t.to(device), m_t.to(device)).cpu().numpy()
    res = pd.DataFrame({'movieId': cands, 'Predicted Rating': np.clip(p, 0.5, 5.0)})
    res = res.nlargest(top_n, 'Predicted Rating').merge(movies_df[['movieId','title','genres']], on='movieId')
    return res[['title','genres','Predicted Rating']].round(2)

for uid in [1, 5, 100]:
    print('='*70)
    print(f'Top 10 Neural CBF Recommendations for User {uid}')
    print('='*70)
    print(recommend_nn(uid, model, user_features, movie_features, movies, train_df).to_string(index=False))
    print()

Top 10 Neural CBF Recommendations for User 1
                                            title                        genres  Predicted Rating
                                  Lamerica (1994)               Adventure|Drama               5.0
             Heidi Fleiss: Hollywood Madam (1995)                   Documentary               5.0
                 Awfully Big Adventure, An (1995)                         Drama               5.0
                           Live Nude Girls (1995)                        Comedy               5.0
In the Realm of the Senses (Ai no corrida) (1976)                         Drama               5.0
                      What Happened Was... (1994) Comedy|Drama|Romance|Thriller               5.0
        Thin Line Between Love and Hate, A (1996)                        Comedy               5.0
                           Denise Calls Up (1995)                        Comedy               5.0
           World of Apu, The (Apur Sansar) (1959)                        

### Analysis

The neural network captures more complex user preferences than TF-IDF CBF because:

1. **Non-linear interactions**: TF-IDF + cosine similarity is linear. The NN learns non-linear genre combinations (e.g., likes Action+Comedy but not Action-only).

2. **Richer features**: Uses release year and movie popularity alongside genres. TF-IDF only uses genre text.

3. **Learned embeddings**: Dense layers compress features into latent representations capturing patterns not obvious from raw features.

4. **Direct optimization**: The NN optimizes MSE on actual ratings. TF-IDF cosine similarity is unsupervised.

However, the NN needs more data and is more prone to overfitting on small datasets. On larger datasets the advantage grows significantly.